In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd

paths = {
    "PERSONAS": "/content/drive/MyDrive/PROYECTO GRADO FILES/Censo 2018 y Geografía/76/CNPV2018_5PER_A2_76.CSV",
    "HOGARES":  "/content/drive/MyDrive/PROYECTO GRADO FILES/Censo 2018 y Geografía/76/CNPV2018_2HOG_A2_76.CSV",
    "VIVIENDAS":"/content/drive/MyDrive/PROYECTO GRADO FILES/Censo 2018 y Geografía/76/CNPV2018_1VIV_A2_76.CSV",
    "FALLECIDOS":"/content/drive/MyDrive/PROYECTO GRADO FILES/Censo 2018 y Geografía/76/CNPV2018_3FALL_A2_76.CSV",
    "MGN":      "/content/drive/MyDrive/PROYECTO GRADO FILES/Censo 2018 y Geografía/76/CNPV2018_MGN_A2_76.CSV",
}

def read_df(path):
    return pd.read_csv(path, sep=",", encoding="latin-1", low_memory=False)


In [ ]:
def filter_cali(df):
    dpto = df["U_DPTO"].astype(str).str.zfill(2)
    mpio = df["U_MPIO"].astype(str).str.zfill(3)
    return df[(dpto == "76") & (mpio == "001")].copy()

dfs_cali = {}

for name, path in paths.items():
    df = read_df(path)
    df_c = filter_cali(df)
    dfs_cali[name] = df_c
    print(name, df.shape, "-> Cali:", df_c.shape)


PERSONAS (3789874, 48) -> Cali: (1822869, 48)
HOGARES (1267729, 13) -> Cali: (612205, 13)
VIVIENDAS (1449853, 30) -> Cali: (690604, 30)
FALLECIDOS (26656, 11) -> Cali: (11929, 11)
MGN (1449853, 14) -> Cali: (690604, 14)


In [ ]:
df_tmp = read_df(paths["PERSONAS"])
df_tmp["U_MPIO"].astype(str).str.zfill(3).value_counts().head(20)


,count
U_MPIO,
001,1822869
520,302642
109,258445
834,187159
364,131806
147,118803
111,115821
892,95040
130,84661


In [ ]:
def norm_keys(df):
    df["U_DPTO"] = df["U_DPTO"].astype(str).str.zfill(2)
    df["U_MPIO"] = df["U_MPIO"].astype(str).str.zfill(3)
    df["COD_ENCUESTAS"] = df["COD_ENCUESTAS"].astype(str)
    df["U_VIVIENDA"] = df["U_VIVIENDA"].astype(str)
    return df

for k in dfs_cali:
    dfs_cali[k] = norm_keys(dfs_cali[k])


In [ ]:
df_PH = dfs_cali["PERSONAS"].merge(
    dfs_cali["HOGARES"],
    left_on=["U_DPTO","U_MPIO","UA_CLASE","COD_ENCUESTAS","U_VIVIENDA","P_NROHOG"],
    right_on=["U_DPTO","U_MPIO","UA_CLASE","COD_ENCUESTAS","U_VIVIENDA","H_NROHOG"],
    how="left",
    suffixes=("", "_H")
)

df_PH.shape


(1822869, 56)

In [ ]:
df_PHV = df_PH.merge(
    dfs_cali["VIVIENDAS"],
    on=["U_DPTO","U_MPIO","UA_CLASE","COD_ENCUESTAS","U_VIVIENDA"],
    how="left",
    suffixes=("", "_V")
)

df_PHV.shape


(1822869, 81)

In [ ]:
df_master = df_PHV.merge(
    dfs_cali["MGN"],
    on=["U_DPTO","U_MPIO","UA_CLASE","COD_ENCUESTAS","U_VIVIENDA"],
    how="left"
)

df_master.shape


(1822869, 90)

In [ ]:
df_master.isna().mean().sort_values(ascending=False).head(15)
df_master["P_SEXO"].value_counts(dropna=False)
df_master["P_EDADR"].value_counts(dropna=False).sort_index()


,count
P_EDADR,
1,97959
2,107548
3,118039
4,136581
5,160271
6,153926
7,139688
8,136399
9,119786


In [ ]:
geo_cols = ["U_DPTO","U_MPIO","UA_CLASE","UA1_LOCALIDAD","U_SECT_URB","U_SECC_URB","U_MZA"]
demo_cols = ["P_SEXO","P_EDADR","P_PARENTESCOR"]
edu_cols = ["P_ALFABETA","PA_ASISTENCIA","P_NIVEL_ANOSR"]
salud_cols = ["P_ENFERMO","PA_LO_ATENDIERON","PA1_CALIDAD_SERV"]
trab_cols = ["P_TRABAJO"]
vivienda_cols = ["V_MAT_PARED","V_MAT_PISO","V_TIPO_VIV","V_CON_OCUP"]
serv_cols = ["VA_EE","VB_ACU","VC_ALC","VD_GAS","VE_RECBAS","VF_INTERNET"]
hacin_cols = ["H_NRO_CUARTOS","H_NRO_DORMIT","HA_TOT_PER"]
key_cols = ["U_VIVIENDA","P_NROHOG","P_NRO_PER"]

keep_cols = list(set(
    geo_cols + demo_cols + edu_cols + salud_cols +
    trab_cols + vivienda_cols + serv_cols + hacin_cols + key_cols
))

existing = set(df_master.columns)
kept = sorted(existing.intersection(keep_cols))
removed = sorted(existing - set(kept))

print("COLUMNAS QUE SE QUEDAN:", len(kept))
print("COLUMNAS ELIMINADAS:", len(removed))

df_report = pd.DataFrame({
    "columna": kept + removed,
    "estado": ["SE QUEDA"]*len(kept) + ["SE ELIMINA"]*len(removed)
})

df_report["bloque"] = df_report["columna"].map(
    lambda c:
        "Geografía" if c in geo_cols else
        "Demografía" if c in demo_cols else
        "Educación" if c in edu_cols else
        "Salud" if c in salud_cols else
        "Trabajo" if c in trab_cols else
        "Vivienda" if c in vivienda_cols else
        "Servicios" if c in serv_cols else
        "Hacinamiento" if c in hacin_cols else
        "Llaves" if c in key_cols else
        "Otros"
)

display(df_report[df_report.estado == "SE QUEDA"].sort_values(["bloque","columna"]))
display(df_report[df_report.estado == "SE ELIMINA"].sort_values(["bloque","columna"]))


COLUMNAS QUE SE QUEDAN: 33
COLUMNAS ELIMINADAS: 57


,columna,estado,bloque
7,P_EDADR,SE QUEDA,Demografía
12,P_PARENTESCOR,SE QUEDA,Demografía
13,P_SEXO,SE QUEDA,Demografía
4,PA_ASISTENCIA,SE QUEDA,Educación
6,P_ALFABETA,SE QUEDA,Educación
9,P_NIVEL_ANOSR,SE QUEDA,Educación
15,UA1_LOCALIDAD,SE QUEDA,Geografía
16,UA_CLASE,SE QUEDA,Geografía
17,U_DPTO,SE QUEDA,Geografía
18,U_MPIO,SE QUEDA,Geografía


,columna,estado,bloque
33,COD_DANE_ANM,SE ELIMINA,Otros
34,COD_ENCUESTAS,SE ELIMINA,Otros
35,CONDICION_FISICA,SE ELIMINA,Otros
36,HA_NRO_FALL,SE ELIMINA,Otros
37,H_AGUA_COCIN,SE ELIMINA,Otros
38,H_DONDE_PREPALIM,SE ELIMINA,Otros
39,H_NROHOG,SE ELIMINA,Otros
40,L_EXISTEHOG,SE ELIMINA,Otros
41,L_TIPO_INST,SE ELIMINA,Otros
42,L_TOT_PERL,SE ELIMINA,Otros


In [ ]:
df_master_reducido = df_master.loc[:, kept].copy()
print("Original:", df_master.shape)
print("Reducido:", df_master_reducido.shape)


Original: (1822869, 90)
Reducido: (1822869, 33)


In [ ]:
output_path = "/content/drive/MyDrive/PROYECTO GRADO FILES/Censo 2018 y Geografía/cali_base_consolidada_reducida.csv.gz"

df_master_reducido.to_csv(output_path, index=False, compression="gzip")

print("Archivo guardado en:", output_path)



Archivo guardado en: /content/drive/MyDrive/PROYECTO GRADO FILES/Censo 2018 y Geografía/cali_base_consolidada_reducida.csv.gz
